# ATLAS — Batch FMNIST @107M (pendentes + campeões)

Roda no **Google Colab A100/H100** os experimentos que faltam do batch CPU @107M, com a **mesma config** (`width_mult=16`, `hidden_dim=2048`, 1000 steps).

**Como usar:**
1. Runtime → Change runtime type → **GPU (A100 ou H100)**
2. Run all (~1–2 h para 9 pendentes + 2 campeões)

**Já concluídos na CPU (não reroda aqui):**
- exp-100m-baseline: **86,12%**
- exp-100m-001/002 warmup+cosine: **89,78% / 89,93%** (líder atual)
- exp-100m-003 local_blend: **89,29%**

**Este notebook roda:**
| ID | Mecanismo |
|----|----------|
| exp-100m-004 … 012 | pendentes do batch CPU |
| exp-100m-010 | blend+LS (rerun — log perdido) |
| exp-100m-ms | **multi_scale + LS + wc** (campeão) |
| exp-100m-tri | **tri_scale + LS + wc** (recorde) |

Copie o JSON final para `experiments_log.jsonl` no repo ou abra um PR.

In [ ]:
# @title 1. GPU
import torch
assert torch.cuda.is_available(), "Ative GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("PyTorch:", torch.__version__)

In [ ]:
# @title 2. Clone repo ATLAS
import os, subprocess, sys

REPO = "https://github.com/Revenn0/0111.git"
BRANCH = "cursor/promissora-scale-c396"

if not os.path.isdir("0111"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, "0111"], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd="0111", check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd="0111", check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd="0111", check=True)

os.chdir("0111")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision"], check=False)
print("OK:", os.getcwd())

In [ ]:
# @title 3. Verificar escala ~107M params
import json, subprocess

out = subprocess.check_output([
    sys.executable, "-c",
    "from train_baseline import build_model, TrainConfig; "
    "c=TrainConfig(width_mult=16.0, hidden_dim=2048); "
    "n=sum(p.numel() for p in build_model(c).parameters()); print(n)"
], text=True).strip()
params_m = int(out) / 1e6
print(f"Params: {params_m:.2f}M (alvo ~107M)")
assert 100 <= params_m <= 115, f"Escala fora da faixa: {params_m:.1f}M"

In [ ]:
# @title 4. Rodar experimentos pendentes @107M (GPU)
import json, subprocess, sys, time
from pathlib import Path
from datetime import datetime, timezone

SCALE = [
    "--width_mult", "16.0", "--hidden_dim", "2048",
    "--batch_size", "64", "--lr", "0.001", "--weight_decay", "1e-4",
    "--device", "cuda", "--amp", "--num_workers", "2",
    "--steps", "1000", "--eval_every", "250", "--log_every", "200",
]
WC = ["--warmup_steps", "100", "--scheduler", "cosine"]
LS = ["--label_smoothing", "0.1"]
SEEDS = 3
RESULT_DIR = Path("results/100m_colab")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# (id, name, novelty, lesson, extra_args)
EXPERIMENTS = [
    ("exp-100m-004", "p018_blend_100m", "NOVEL exp-018", "local_blend", ["--mixing", "local_blend"]),
    ("exp-100m-005", "p021_k5_100m", "NOVEL exp-021", "k5 blend", ["--mixing", "local_blend_k5"]),
    ("exp-100m-006", "p022_laplacian_100m", "NOVEL exp-022", "laplacian", ["--mixing", "laplacian_blend"]),
    ("exp-100m-007", "p024_blend_wc_100m", "RECOMB exp-024", "blend+wc", ["--mixing", "local_blend"] + WC),
    ("exp-100m-008", "p025_blend_ls_100m", "RECOMB exp-025", "blend+LS", ["--mixing", "local_blend"] + LS),
    ("exp-100m-009", "p026_k5_100m", "NOVEL exp-026", "k5", ["--mixing", "local_blend_k5"]),
    ("exp-100m-010", "p027_blend_ls_100m", "RECOMB exp-027", "blend+LS campeão", ["--mixing", "local_blend"] + LS),
    ("exp-100m-011", "p028_laplacian_100m", "NOVEL exp-028", "laplacian", ["--mixing", "laplacian_blend"]),
    ("exp-100m-012", "p032_k5_ls_100m", "RECOMB exp-032", "k5+LS", ["--mixing", "local_blend_k5"] + LS),
    ("exp-100m-ms", "multi_scale_ls_wc_100m", "NOVEL exp-042", "multi_scale campeão", ["--mixing", "multi_scale_blend"] + LS + WC),
    ("exp-100m-tri", "tri_scale_ls_wc_100m", "NOVEL exp-053", "tri_scale recorde", ["--mixing", "tri_scale_blend"] + LS + WC),
]

CPU_BASELINE = 0.8612  # régua CPU já medida
CPU_LEADER = 0.8993    # exp-100m-002
LOG_ENTRIES = []

def run_one_seed(extra, seed):
    out = RESULT_DIR / f"seed{seed}.json"
    cmd = [sys.executable, "train_baseline.py"] + SCALE + extra + [
        "--seed", str(seed), "--result_path", str(out)
    ]
    subprocess.run(cmd, check=True)
    return json.loads(out.read_text())

for eid, name, nov, lesson, extra in EXPERIMENTS:
    print(f"\n{'='*60}\n{eid} | {lesson}\n{'='*60}")
    t0 = time.time()
    per_seed = []
    for seed in [1000, 1001, 1002]:
        r = run_one_seed(extra, seed)
        per_seed.append(r)
        print(f"  seed {seed}: {r['best_val_acc']*100:.2f}%  ({r['steps_per_sec']:.1f} steps/s)")
    accs = [r["best_val_acc"] for r in per_seed]
    mean = sum(accs) / len(accs)
    std = (sum((a-mean)**2 for a in accs)/max(1,len(accs)-1))**0.5 if len(accs)>1 else 0
    entry = {
        "id": eid,
        "name": name,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "source": "colab_a100_100m",
        "seeds": [1000,1001,1002],
        "result": {
            "n_seeds": 3,
            "best_val_acc_mean": mean,
            "best_val_acc_std": std,
            "wall_time_s_mean": sum(r["wall_time_s"] for r in per_seed)/3,
            "steps_per_sec_mean": sum(r["steps_per_sec"] for r in per_seed)/3,
            "per_seed": per_seed,
        },
        "delta_vs_cpu_baseline_pp": round((mean - CPU_BASELINE)*100, 2),
        "delta_vs_cpu_leader_pp": round((mean - CPU_LEADER)*100, 2),
        "novelty_note": nov,
        "lesson": lesson,
    }
    LOG_ENTRIES.append(entry)
    print(f"  >> média {mean*100:.2f}% ± {std*100:.2f}%  Δbase={entry['delta_vs_cpu_baseline_pp']:+.2f}pp  ({time.time()-t0:.0f}s)")

out_path = Path("experiments_log_100m_colab.jsonl")
with out_path.open("w") as f:
    for e in LOG_ENTRIES:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
print(f"\nSalvo: {out_path.resolve()}")

In [ ]:
# @title 5. Ranking final (GPU @107M + referência CPU)
import json
from pathlib import Path

CPU_REF = [
    ("exp-100m-002", "warmup+cosine (CPU)", 0.8993, "líder CPU"),
    ("exp-100m-001", "warmup+cosine (CPU)", 0.8978, "CPU"),
    ("exp-100m-003", "local_blend (CPU)", 0.8929, "CPU"),
    ("exp-100m-baseline", "baseline (CPU)", 0.8612, "régua"),
]

rows = [(e["id"], e["lesson"], e["result"]["best_val_acc_mean"], "GPU Colab") for e in LOG_ENTRIES]
rows += [(a,b,c,d) for a,b,c,d in CPU_REF]
rows.sort(key=lambda x: -x[2])

print(f"{'Rank':<5} {'ID':<18} {'Acc':>8} {'Δ base':>8}  {'Fonte':<10} {''}")
print("-"*70)
for i, (eid, label, acc, src) in enumerate(rows, 1):
    d = (acc - 0.8612) * 100
    print(f"{i:<5} {eid:<18} {acc*100:7.2f}% {d:+7.2f}pp  {src:<10} {label}")

best_gpu = max(LOG_ENTRIES, key=lambda e: e["result"]["best_val_acc_mean"])
bacc = best_gpu["result"]["best_val_acc_mean"]
print(f"\nMelhor GPU: {best_gpu['id']} = {bacc*100:.2f}%")
if bacc > 0.8993:
    print("🏆 BATEU o líder CPU (warmup+cosine 89.93%)!")
elif bacc > 0.8993 - 0.001:
    print("≈ Empate técnico com líder CPU")
else:
    print(f"Ainda abaixo do líder CPU por {(0.8993-bacc)*100:.2f} pp")

In [ ]:
# @title 6. Download resultados
from google.colab import files
files.download("experiments_log_100m_colab.jsonl")